# Lens formation and spot-diagram evolution
### A fresh forward run of the single-face model from the linked viewer

This notebook shows **computed fluid motion**, not interpolated target geometry.
The physical sequence starts from the unforced liquid surface with zero mean velocity,
then applies the fixed holding drive and evolves for **0.2 s**.
Two new runs use time steps **1.25 ms and 0.625 ms** without changing physical commands.
The animation uses the finer run.

**This is the earlier one-free-surface apparatus, not the two-face stationary model.**
The outer cylinder, base and source supports are shown in 3D below; the transient is shown in 2D.
A prescribed spherical wavefront inside the liquid is refracted at its moving surface.
The spot diagram stays at one fixed detector plane; it is not repeatedly refocused.

Implemented: axisymmetric ALE flow, inertia, convection, nonlinear capillarity, gravity,
instantaneous cycle-mean acoustics and declared bulk absorption forcing.
Acoustic ring-up at switch-on is not resolved; the phasor response supplies the slow-flow load.
Not established: thermal feedback, wall-layer mean streaming, calibrated material acoustics,
three-dimensional stability, curing, or experimental precision.
[Model limitations](../docs/numerical-stability.md) · [Reproduction background](../docs/stable-cartesian-reproduction.md)

**New below: the two-face setup, showing both bottom and top interfaces.** Its saved
solutions are stationary. The later formation, spot and sound sections still belong
to the separate single-face model; they must not be interpreted as two-face dynamics.


## 0. Both bottom and top surfaces: the two-face setup

The central lens fluid is bounded by **two deformable liquid interfaces**, surrounded
by lower and upper fluids inside the cylinder. The rigid end boundaries are **not**
these lens surfaces. The blue front/bottom interface is near z = −1 mm; the orange
back/top interface is near z = +1 mm.

The saved shared drive has 448 ideal source regions at 7.2 MHz. These are boundary
source distributions, not a fabricated array of point speakers. We can display and
rotate this setup directly from the saved solutions.

**Camera rotation is not physical motion.** The selector changes surface discretization
at fixed commands, without interpolating states. No two-face formation trajectory,
joint two-surface spot evolution, or associated sound is inferred from the single-face
sections later in this notebook. The refined solution fails 10 nm on both faces.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from acoustic_freeform.dual.presentation import load_setup, setup_figures, interactive_setup
from acoustic_freeform.provenance import capture_execution

DUAL_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "pyproject.toml").exists() and (p / "src/acoustic_freeform").is_dir())
DUAL_OUT = DUAL_ROOT / "artifacts/notebooks/05_formation_and_spots/two-face-setup"
DUAL_OUT.mkdir(parents=True, exist_ok=True)
dual_records, dual_hashes = load_setup(DUAL_ROOT)
dual_3d, dual_profiles = setup_figures(dual_records)
dual_3d.savefig(DUAL_OUT / "both-interfaces-3d.png", dpi=150, bbox_inches="tight")
dual_profiles.savefig(DUAL_OUT / "both-interfaces-profiles.png", dpi=150, bbox_inches="tight")
plt.show()
dual_html = interactive_setup(dual_records)
(DUAL_OUT / "setup.html").write_text(
    '<!doctype html><meta charset="utf-8"><title>Both lens interfaces</title>' + dual_html)
display(HTML(dual_html))
np.savez_compressed(DUAL_OUT / "geometry-si.npz",
    radius_m=dual_records[0]["r_m"],
    interface_z_m=np.stack([r["z_m"] for r in dual_records]),
    target_z_m=np.stack([r["target_z_m"] for r in dual_records]),
    surface_elements=np.array([r["cfg"].surface_elements for r in dual_records]))
for path in (DUAL_ROOT / "notebooks/05_formation_and_spots.ipynb",
             DUAL_ROOT / "src/acoustic_freeform/dual/presentation.py"):
    dual_hashes[str(path.relative_to(DUAL_ROOT))] = hashlib.sha256(path.read_bytes()).hexdigest()
capture_execution(DUAL_OUT)
(DUAL_OUT / "config.json").write_text(json.dumps({
    "operation": "Saved two-face setup reconstruction and camera rotation",
    "input_sha256": dual_hashes, "physical_time_axis": False,
    "geometry_interpolation_between_states": False, "geometry_scale": 1,
    "display_units": "mm; files SI", "new_physical_simulation": False
}, indent=2) + "\n")
(DUAL_OUT / "validation.json").write_text(json.dumps({
    "source_commands_identical": True, "interfaces_do_not_cross": True,
    "rims_match_declared_interface_levels": True,
    "elements": [r["cfg"].surface_elements for r in dual_records],
    "final_front_back_max_error_m": [f["max_error_m"] for f in dual_records[-1]["result"]["faces"]],
    "physical_accuracy_certified": False, "formation_simulated": False
}, indent=2) + "\n")
(DUAL_OUT / "validation-report.md").write_text(
    "# Two-face setup display\n\n"
    "Reconstructed both interfaces from saved coefficients at 52, 80 and 104 surface "
    "elements. Exact shared commands, noncrossing surfaces and pinned rims are checked. "
    "SI geometry and source hashes are retained. Camera rotation and discrete state "
    "selection are visualization, not physical time. No new dynamics or optical "
    "propagation through the pair is claimed. The 104-element state fails 10 nm "
    "on both faces. Cylinder and rigid end outlines have no invented wall thickness "
    "or transducer hardware.\n")

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, Image, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/acoustic_freeform").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from acoustic_freeform.lens.formation_presentation import (
    load_formation, analyze_formation, apparatus_figure, formation_animation)
from acoustic_freeform.provenance import capture_execution

CAMPAIGN = ROOT / "artifacts/formation-notebook-2026-09-23"
OUT = ROOT / "artifacts/notebooks/05_formation_and_spots"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 95, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
source = ROOT / "artifacts/cartesian-maintenance-2026-09-06/operating-state"
commands = []
# Existing completed outputs are read, never overwritten. On a clean checkout,
# this cell actually runs the forward solver; allow several minutes per run.
for name, step in [("dt00125", "0.00125"), ("dt000625", "0.000625")]:
    destination = CAMPAIGN / name
    command = [sys.executable, "-c", "from acoustic_freeform.cli import main; main()", "evolve", str(source),
               "--controller", str(source / "hold-model"), "--initial", "rest_fixed",
               "--step", step, "--duration", "0.2", "--out", str(destination)]
    commands.append(command)
    if not (destination / "report.json").exists():
        if destination.exists() and any(destination.iterdir()):
            raise RuntimeError(f"Partial or active run at {destination}; inspect before resuming.")
        subprocess.run(command, cwd=ROOT, check=True)
    print(f"Completed forward data: {destination.relative_to(ROOT)}")
coarse, fine = [load_formation(CAMPAIGN / name) for name in ("dt00125", "dt000625")]
cfg, space = fine["cfg"], fine["space"]

## 1. The apparatus in three dimensions

The plot uses equal scale in millimetres. Removing the front half of the wall is
a **visual cutaway**, not an opening in the simulated chamber.
Orange markings indicate the imposed acoustic velocity bands on the inner wall.
They are not a design for real piezoelectric hardware or an invented backing layer.
The axisymmetric solver controls 16 axial rows; configured azimuthal sectors share
each row's command. The base has its configured thickness; no extra retaining ring is added.

In [ ]:
fig = apparatus_figure(fine)
fig.savefig(OUT / "apparatus-3d.png", dpi=180, bbox_inches="tight")
fig.savefig(OUT / "apparatus-3d.svg", bbox_inches="tight")
plt.show()
display(Markdown(
    f"**Chamber:** radius {cfg.radius_m*1e3:g} mm, depth {cfg.depth_m*1e3:g} mm; "
    f"clear pupil radius {cfg.clear_radius_m*1e3:g} mm. "
    f"**Actuation:** {cfg.array_rows} independent row commands at {cfg.frequency_hz/1e6:g} MHz.\n\n"
    "The source wavefront is specified inside the liquid; a complete external illumination "
    "system and base-window optics are not modeled."))

## 2. Reconstruct the computed surfaces and optical rays

The height error is measured against the **exact Cartesian surface** in fixed laboratory
coordinates, without piston/tilt fitting. Maxima use radial derivative-root searches and
endpoints, checked against dense samples; this is not an interval-arithmetic certificate.

At each saved state, vector Snell refraction gives ray intercepts at the same detector.
Equal-area pupil radii with distributed azimuths produce the axisymmetric 2D spot diagram.
It is a **geometrical ray diagram**, not a diffraction PSF or a measured camera image.
Shapes are cycle-mean interfaces; carrier-frequency ripples are not resolved.
Tracing rays through the mean interface is not a full cycle-averaged optical-intensity calculation.
Optical indices are fixed; pressure- and temperature-induced index changes are not included.
The fixed target is not substituted for the evolving surface.

In [ ]:
a_coarse, a = analyze_formation(coarse), analyze_formation(fine)
assert len(a["time_s"]) == 321 and np.isclose(a["time_s"][-1], .2)
np.testing.assert_allclose(coarse["data"]["drive_m_s"][0], fine["data"]["drive_m_s"][0],
                           rtol=0, atol=0)
np.testing.assert_allclose(coarse["data"]["initial"], fine["data"]["initial"], rtol=0, atol=0)
for key, value in coarse["report"]["configuration"].items():
    if key != "step_s":
        assert fine["report"]["configuration"][key] == value, key
# Independent analytic limit: the exact Cartesian surface focuses at the fixed detector.
r = np.linspace(0, cfg.clear_radius_m, 503)
z = cfg.diopter.sag(r)
fr, fz = cfg.diopter.fermat_gradient(r, z)
direction = cfg.diopter.refract(r, z, -fr/fz)
exact_hits = r + (cfg.focal_distance_m-z)*direction[:, 0]/direction[:, 1]
assert np.max(abs(exact_hits)) < 1e-11
# Cross-check the fresh ray traces with the simulation's saved ray diagnostics.
saved_rms = np.array([row["rms_spot_at_target_m"] for row in fine["report"]["history"]])
np.testing.assert_allclose(a["spot_rms_m"], saved_rms, rtol=1e-10, atol=1e-12)
np.savez_compressed(OUT / "formation-analysis-si.npz", **a)
print(f"Fixed detector: z = {a['detector_z_m']*1e3:.6f} mm above the rim.")
print(f"Final maximum height error: {a['maximum_height_error_m'][-1]*1e9:.4f} nm.")
print(f"Final geometric spot RMS: {a['spot_rms_m'][-1]*1e6:.6f} µm.")

## 3. Watch formation, error and the spot diagram together

Press **Play**. The figure title and both time cursors show the physical time.
Every displayed surface and spot diagram comes from the **same saved fluid state**.
Playback shows every second fine time step (1.25 ms spacing), with no state interpolation.
The error histories use all 321 saved states. Gray curves show the complete recorded
history; the moving cursors select the state currently displayed above.

The full spot view has a fixed scale covering the entire trajectory.
The separate final-state zoom also has a fixed scale and explicitly reports clipped rays.
The surface panel has equal geometric scale; its shape may look nearly unchanged while
nanometre-scale errors remain visible in the error panel. Playback is slowed for inspection.

In [ ]:
html, frame_indices, snapshots = formation_animation(fine, a, frame_stride=2)
standalone = (
    '<!doctype html><meta charset="utf-8"><title>Lens formation and spot evolution</title>'
    '<h1>Single-face forward formation and optical spots</h1>'
    '<p>Fresh 0–0.2 s ALE simulation. Not the newer two-face model. '
    'Saved fluid states; fixed detector; geometric rays, not a diffraction PSF. '
    'Physical time is displayed in milliseconds. Playback is slowed.</p>' + html)
(OUT / "formation-and-spots.html").write_text(standalone)
for k, png in snapshots:
    (OUT / f"formation-frame-{k:03d}.png").write_bytes(png)
display(HTML(html))

## 4. Time-step sensitivity — unchanged commands

The same physical problem is run twice, not re-optimized.
Comparing common physical times exposes discretization sensitivity.
Two step sizes alone do **not** establish a convergence order, a precisely converged
settling time, or a 10 nm physical accuracy claim. Spatial and material errors remain separate.
In particular, agreement at the final state does not validate the early transient.

In [ ]:
np.testing.assert_allclose(a_coarse["time_s"], a["time_s"][::2], rtol=0, atol=1e-15)
# Compare actual surface shapes, not just differences between scalar maxima.
shape_difference = np.max(abs(a_coarse["surface_height_m"] - a["surface_height_m"][::2]), axis=1)
ray_difference = abs(a_coarse["spot_rms_m"] - a["spot_rms_m"][::2])
fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
for values, style, label in [(a_coarse, "--", "dt = 1.25 ms"), (a, "-", "dt = 0.625 ms")]:
    axes[0].semilogy(values["time_s"]*1e3, values["maximum_height_error_m"]*1e9, style, label=label)
    axes[1].semilogy(values["time_s"]*1e3, values["spot_rms_m"]*1e6, style, label=label)
axes[0].axhline(10, color="k", ls=":", label="10 nm")
axes[0].set(ylabel="Max exact-target height error (nm)")
axes[1].set(ylabel="Fixed-detector ray RMS (µm)")
axes[2].semilogy(a_coarse["time_s"]*1e3, np.maximum(shape_difference*1e9, 1e-7))
axes[2].set(ylabel="Max full-radius inter-run shape difference (nm)",
            title="Plot floor 1e-7 nm at identical initial state")
for ax in axes:
    ax.set_xlabel("Physical time (ms)")
    ax.grid(alpha=.2)
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
fig.savefig(OUT / "time-step-comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print(f"Maximum time-step shape discrepancy: {shape_difference.max()*1e9:.3f} nm.")
print(f"Final time-step shape discrepancy: {shape_difference[-1]*1e9:.6f} nm.")
print("These discrepancies are sensitivity measurements, not error bounds.")

## 5. Numerical audit and provenance

The forward solver independently recomputes acoustic force and distributed absorption
forcing on the evolving geometry. Its supplied frozen radiation Jacobian is an implicit
stepping aid—not the physical trajectory or a full coupled stability certificate.

Original artifacts are preserved. The new runs and this analysis have separate provenance.
The older solver saves dimensionless surface coefficients, explicitly scaled by chamber
radius here; the analysis export uses **SI geometry, time, error and spot coordinates**.

Material constants include unmeasured sound-speed/attenuation assumptions.
The experiment remains a numerical demonstration under that model, not a calibrated
NOA61 apparatus, a cured lens, or validation of the latest two-face design.

In [ ]:
hashes = {}
for run in (coarse, fine):
    for filename in ("trajectory.npz", "report.json", "configuration.json"):
        path = run["directory"] / filename
        hashes[str(path.relative_to(ROOT))] = hashlib.sha256(path.read_bytes()).hexdigest()
for filename in ("stationary.npz", "configuration.json", "hold-model/controller.npz",
                 "hold-model/report.json", "control-model/linearization.npz"):
    path = source / filename
    hashes[str(path.relative_to(ROOT))] = hashlib.sha256(path.read_bytes()).hexdigest()
for path in (ROOT / "notebooks/05_formation_and_spots.ipynb",
             ROOT / "src/acoustic_freeform/lens/formation_presentation.py"):
    hashes[str(path.relative_to(ROOT))] = hashlib.sha256(path.read_bytes()).hexdigest()
previous_path = ROOT / "artifacts/cartesian-maintenance-2026-09-06/step-formation-dt00125/trajectory.npz"
hashes[str(previous_path.relative_to(ROOT))] = hashlib.sha256(previous_path.read_bytes()).hexdigest()
with np.load(previous_path, allow_pickle=False) as previous:
    np.testing.assert_array_equal(previous["times_s"], coarse["data"]["times_s"])
    replay_difference = float(np.max(abs(previous["coefficients"] - coarse["data"]["coefficients"])))
volume_error = max(abs(row["volume_error_m3"]) for row in fine["report"]["history"])
audit = {
    "fresh_forward_runs": 2, "frames_coarse": len(a_coarse["time_s"]), "frames_fine": len(a["time_s"]),
    "coarse_replay_max_dimensionless_coefficient_difference": replay_difference,
    "animation_frames": len(frame_indices), "same_commands_exactly": True,
    "same_initial_condition": True, "exact_cartesian_ray_limit_max_m": float(np.max(abs(exact_hits))),
    "independent_optical_reconstruction_matches": True,
    "maximum_volume_error_m3": volume_error,
    "full_radius_time_step_max_shape_difference_m": float(shape_difference.max()),
    "final_full_radius_time_step_shape_difference_m": float(shape_difference[-1]),
    "final_maximum_height_error_m": float(a["maximum_height_error_m"][-1]),
    "final_spot_rms_m": float(a["spot_rms_m"][-1]),
    "time_convergence_certified": False, "physical_accuracy_certified": False,
    "model": "Earlier single-interface ALE; not simultaneous two-face formation"
}
capture_execution(OUT)
(OUT / "config.json").write_text(json.dumps({"operation": "Fresh formation-run notebook",
    "commands": commands, "input_sha256": hashes, "frame_indices": frame_indices.tolist(),
    "detector_z_m": a["detector_z_m"], "frame_interpolation": False}, indent=2) + "\n")
(OUT / "validation.json").write_text(json.dumps(audit, indent=2) + "\n")
(OUT / "validation-report.md").write_text(
    "# Formation notebook validation\n\n"
    "Two fresh fixed-command ALE runs, 1.25 ms and 0.625 ms steps, each to 0.2 s. "
    "The animation uses saved fine-run surface states and independently traced rays "
    "at a fixed detector. No geometric interpolation or refocusing.\n\n"
    "Checks: identical drives and initial states; exact Cartesian optical limit; "
    "reconstructed spot RMS versus saved diagnostics; time-step sensitivity; "
    "recorded volume error. Values are in validation.json. "
    "Two step sizes do not establish time convergence or experimental accuracy.\n\n"
    "The three-dimensional apparatus depicts only the declared base, wall, "
    "liquid surface and ideal wall source supports. The cutaway is visual only.\n\n"
    "This is the earlier single-face model, not two-face formation. "
    "Thermal feedback, wall-layer mean streaming, material calibration, curing "
    "and full three-dimensional stability are not validated.\n")
# Each new solver run gets an explicit presentation/reproduction validation report.
for run in (coarse, fine):
    path = run["directory"] / "validation-report.md"
    if not path.exists():
        path.write_text(
            "# Fresh single-face formation run\n\n"
            "Executed nonlinear ALE forward dynamics at fixed physical holding commands. "
            "See report.json for implemented physics, diagnostics and captured provenance. "
            "No source re-optimization. Companion analysis and time-step comparison: "
            "artifacts/notebooks/05_formation_and_spots/.\n\n"
            "This run does not establish a converged settling time, full stability, "
            "physical 10 nm accuracy or simultaneous two-face formation.\n")
display(Markdown("**Executed and audited.** Standalone animation, static figures, SI data "
                 "and validation are in `artifacts/notebooks/05_formation_and_spots/`."))
print(json.dumps(audit, indent=2))

## 6. Hear a diagnostic sonification, synchronized with formation

**This is not a recording of the apparatus.** Its modeled 1 MHz drive is ultrasonic.
The saved trajectory contains pressure amplitudes, not a microphone waveform or phase history.
We map the spatial peak cavity-pressure envelope to a synthetic **440 Hz tone**, stretch
0.2 s of physical time to **17.6 s** of playback, and normalize the volume to 12% of digital
full scale. Short fades prevent playback clicks. This does not predict audible structural
noise, sound transmitted into air, or sound-pressure level.

Use the audio player's Play button below to drive the animation above. Seeking the audio
also selects the corresponding saved surface/spot frame. The original animation controls
operate silently and stop audio. No sound starts automatically.

In [ ]:
from acoustic_freeform.lens.sonification import (
    pressure_sonification, wav_bytes, synchronized_audio_html)
SOUND = OUT / "sonification"
SOUND.mkdir(exist_ok=True)
peak_pressure = np.array([row["peak_acoustic_pressure_pa"] for row in fine["report"]["history"]])
playback_step_s = .110
physical_steps = np.diff(a["time_s"][frame_indices])
np.testing.assert_allclose(physical_steps, physical_steps[0], rtol=0, atol=1e-14)
slowdown = playback_step_s / physical_steps[0]
samples, sound_metadata = pressure_sonification(a["time_s"], peak_pressure, slowdown=slowdown)
sound_metadata["model_carrier_hz"] = cfg.frequency_hz
sound_metadata["input_trajectory_sha256"] = hashes[str((fine["directory"] / "trajectory.npz").relative_to(ROOT))]
sound_metadata["input_report_sha256"] = hashes[str((fine["directory"] / "report.json").relative_to(ROOT))]
audio_data = wav_bytes(samples, sound_metadata["sample_rate_hz"])
(SOUND / "pressure-envelope-sonification.wav").write_bytes(audio_data)
sound_metadata["wav_sha256"] = hashlib.sha256(audio_data).hexdigest()
(SOUND / "config.json").write_text(json.dumps(sound_metadata, indent=2) + "\n")
np.savez_compressed(SOUND / "envelope-si.npz", time_s=a["time_s"], peak_pressure_pa=peak_pressure)
capture_execution(SOUND)
audio_html = synchronized_audio_html(html, audio_data, playback_step_s)
(OUT / "formation-and-spots.html").write_text(standalone + audio_html)
display(HTML(audio_html))
audio_validation = {
    "finite_samples": bool(np.all(np.isfinite(samples))),
    "sample_count": len(samples), "playback_duration_s": len(samples)/48000,
    "peak_absolute_sample": float(np.max(abs(samples))),
    "no_digital_clipping": bool(np.max(abs(samples)) < 1),
    "same_physical_timeline_as_animation": True,
    "actual_apparatus_audio_predicted": False,
    "acoustic_phase_history_available": False
}
assert audio_validation["finite_samples"] and audio_validation["no_digital_clipping"]
(SOUND / "validation.json").write_text(json.dumps(audio_validation, indent=2) + "\n")
(SOUND / "validation-report.md").write_text(
    "# Sonification scope and validation\n\n"
    "Synthetic 440 Hz tone with an envelope from saved spatial peak cavity-pressure "
    "amplitudes. Physical time is stretched 88 times; digital gain and short fades "
    "are arbitrary playback choices. Not a microphone recording, a phase-preserving "
    "frequency shift or a prediction of audible apparatus noise.\n\n"
    "The original model carrier is 1 MHz. Samples are finite and below clipping. "
    "Unit tests verify tone frequency, duration, amplitude mapping and WAV decoding. "
    "Browser checks verify audio-clock synchronization separately.\n")
print(json.dumps(audio_validation, indent=2))